In [0]:
from pyspark.sql.functions import *
from pyspark.sql.types import *
import os
import sys
project_pth=os.path.join(os.getcwd(),'..','..')
sys.path.append(project_pth)

from utils.Transformations import reusable

**DimUser**


## AUTOLOADER

In [0]:
df_user = spark.readStream.format("cloudFiles")\
                .option("cloudFiles.format","parquet")\
                .option("cloudFiles.schemaLocation","abfss://silver@storageazureshan.dfs.core.windows.net/DimUser/checkpoint")\
                .option("cloudFiles.schemaEvolutionMode","addNewColumns")\
                .load("abfss://bronze@storageazureshan.dfs.core.windows.net/DimUser")

In [0]:
   query = (df_user.writeStream
       .format("delta")
       .outputMode("append")
       .trigger(availableNow=True)
       .option("checkpointLocation", "abfss://silver@storageazureshan.dfs.core.windows.net/DimUser/checkpoint_write")
       .start("abfss://silver@storageazureshan.dfs.core.windows.net/DimUser/data"))
   query.awaitTermination()

26/10/23 14:46:52 Query 77cd37b1-2424-497b-97f1-118cae782e90 have exception: com.databricks.sql.transaction.tahoe.DeltaAnalysisExceptionWithSubErrors: [DELTA_METADATA_MISMATCH] A metadata mismatch was detected when writing to the Delta table.
- A schema mismatch detected when writing to the Delta table (Table ID: d0198078-69d5-4d50-ba4f-14659f3f8731).
To enable schema migration using DataFrameWriter or DataStreamWriter, please set: '.option("mergeSchema", "true")'.
For other operations, set the session configuration spark.databricks.delta.schema.autoMerge.enabled to "true". See the documentation specific to the operation for details.

Table schema:
root
 |-- user_id: integer (nullable = true)
 |-- user_name: string (nullable = true)
 |-- country: string (nullable = true)
 |-- subscription_type: string (nullable = true)
 |-- start_date: date (nullable = true)
 |-- end_date: date (nullable = true)
 |-- updated_at: timestamp (nullable = true)


Data schema:
root
 |-- user_id: integer (nul

---------------------------------------------------------------------------
StreamingQueryException                   Traceback (most recent call last)
File <command-4515633750417778>, line 7
      1 query = (df_user.writeStream
      2     .format("delta")
      3     .outputMode("append")
      4     .trigger(availableNow=True)
      5     .option("checkpointLocation", "abfss://silver@storageazureshan.dfs.core.windows.net/DimUser/checkpoint_write")
      6     .start("abfss://silver@storageazureshan.dfs.core.windows.net/DimUser/data"))
----> 7 query.awaitTermination()

File /databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/streaming/query.py:96, in StreamingQuery.awaitTermination(self, timeout)
     94 await_termination_cmd = pb2.StreamingQueryCommand.AwaitTerminationCommand()
     95 cmd.await_termination.CopyFrom(await_termination_cmd)
---> 96 self._execute_streaming_query_cmd(cmd)
     97 return None

File /databricks/python/lib/python3.12/site-packages/pyspark/s

In [0]:
display(df_user, checkpointLocation="abfss://silver@storageazureshan.dfs.core.windows.net/DimUser/checkpoint_display_v4")

Checkpointing to abfss://silver@storageazureshan.dfs.core.windows.net/DimUser/checkpoint_display_v4


---------------------------------------------------------------------------
AnalysisException                         Traceback (most recent call last)
File <command-4515633750417779>, line 1
----> 1 display(df_user, checkpointLocation="abfss://silver@storageazureshan.dfs.core.windows.net/DimUser/checkpoint_display_v4")

File <command-4515633750417776>, line 1
----> 1 df_user = spark.readStream.format("cloudFiles")\
      2                 .option("cloudFiles.format","parquet")\
      3                 .option("cloudFiles.schemaLocation","abfss://silver@storageazureshan.dfs.core.windows.net/DimUser/checkpoint")\
      4                 .option("cloudFiles.schemaEvolutionMode","addNewColumns")\
      5                 .load("abfss://bronze@storageazureshan.dfs.core.windows.net/DimUser")

AnalysisException: [STREAMING_OUTPUT_MODE.UNSUPPORTED_OPERATION] Invalid streaming output mode: complete. This output mode is not supported for no streaming aggregations on streaming DataFrames/DataSets

In [0]:
df_user=df_user.withColumn("user_name",upper(col("user_name")))


In [0]:
df_user_obj = reusable()
df_user=df_user_obj.dropColumns(df_user,['_rescued_data'])
df_user=df_user_obj.dropDuplicates(df_user,['user_id'])


In [0]:
df_user.writeStream.format("delta")\
        .outputMode("append")\
        .option("checkpointLocation","abfss://silver@storageazureshan.dfs.core.windows.net/DimUser/chekpoint")\
        .trigger(once=True)\
        .option("path","abfss://silver@storageazureshan.dfs.core.windows.net/DimUser/data")\
        .toTable("spotify_cata.silver.DimUser")

*DimArtist*

In [0]:
df_art = spark.readStream.format("cloudFiles")\
            .option("cloudFiles.format","parquet")\
            .option("cloudFiles.schemaLocation","abfss://silver@storageazureshan.dfs.core.windows.net/DimArtist/checkpoint")\
            .option("cloudFiles.schemaEvolutionMode","addNewColumns")\
            .load("abfss://bronze@storageazureshan.dfs.core.windows.net/DimArtist")

In [0]:
df_art.writeStream.format("delta")\
        .outputMode("append")\
        .option("checkpointLocation","abfss://silver@storageazureshan.dfs.core.windows.net/DimArtist/checkpoint_write2")\
        .trigger(availableNow=True)\
        .start("abfss://silver@storageazureshan.dfs.core.windows.net/DimArtist/data2")

In [0]:
display(spark.read.format("delta").load("abfss://silver@storageazureshan.dfs.core.windows.net/DimArtist/data2"))

artist_id,artist_name,genre,country,updated_at,_rescued_data
1,Lauren Singleton,Rock,Oman,2025-09-26T19:49:55.000Z,null
2,Gregory Mccarty,Classical,Suriname,2025-09-23T19:49:55.000Z,null
3,Adam Stanton,Hip-Hop,Anguilla,2025-09-09T19:49:55.000Z,null
4,Monica Roth,Jazz,Argentina,2025-09-15T19:49:55.000Z,null
5,Calvin Blake,Rock,Greenland,2025-09-18T19:49:55.000Z,null
6,James Lowe,Pop,Saudi Arabia,2025-09-17T19:49:55.000Z,null
7,Jacqueline Evans,Jazz,Syrian Arab Republic,2025-09-10T19:49:55.000Z,null
8,Paul Johnson,Electronic,Bangladesh,2025-09-19T19:49:55.000Z,null
9,Alexa Garcia,Rock,Burkina Faso,2025-09-09T19:49:55.000Z,null
10,Robert Ellis,Rock,Ecuador,2025-09-19T19:49:55.000Z,null


In [0]:
df_art_obj = reusable()

df_art =df_art_obj.dropColumns(df_art,['_rescued_data'])
df_art =df_art_obj.dropDuplicates(df_art,['artist_id'])

In [0]:
df_art.writeStream.format("delta")\
        .outputMode("append")\
        .option("checkpointLocation","abfss://silver@storageazureshan.dfs.core.windows.net/DimArt/chekpoint")\
        .trigger(once=True)\
        .option("path","abfss://silver@storageazureshan.dfs.core.windows.net/DimArt/data")\
        .toTable("spotify_cata.silver.DimArtist")

In [0]:
display(spark.read.format("delta").load("abfss://silver@storageazureshan.dfs.core.windows.net/DimArt/data"))

artist_id,artist_name,genre,country,updated_at
123,Anthony Lane,Classical,Maldives,2025-09-27T19:49:55.000Z
342,Misty Robinson,Jazz,Indonesia,2025-09-12T19:49:55.000Z
373,Alexander Robinson,Electronic,Dominican Republic,2025-10-02T19:49:55.000Z
379,Kimberly Reed,Pop,Vanuatu,2025-09-17T19:49:55.000Z
454,Victoria Williams,Hip-Hop,British Indian Ocean Territory (Chagos Archipelago),2025-10-06T19:49:55.000Z
20,Kelly Brown,Electronic,Nicaragua,2025-09-25T19:49:55.000Z
40,Karen Meyer,Electronic,Malawi,2025-10-05T19:49:55.000Z
164,Michelle Johnson,Pop,United States Minor Outlying Islands,2025-09-18T19:49:55.000Z
429,Kelli Turner,Pop,Latvia,2025-10-07T19:49:55.000Z
470,Matthew Ortiz Jr.,Hip-Hop,Monaco,2025-09-19T19:49:55.000Z


## DimTrack

In [0]:
   df_track = spark.readStream.format("cloudFiles")\
       .option("cloudFiles.format","parquet")\
       .option("cloudFiles.schemaLocation","abfss://silver@storageazureshan.dfs.core.windows.net/DimTrack/checkpoint")\
       .option("cloudFiles.schemaEvolutionMode","addNewColumns")\
       .load("abfss://bronze@storageazureshan.dfs.core.windows.net/DimTrack")

In [0]:
df_track = df_track.withColumn("durationFlag",
    when(col('duration_sec') < 150, "low")
    .when(col('duration_sec') < 300, "medium")
    .otherwise("high"))

df_track = df_track.withColumn("track_name", regexp_replace(col('track_name'), '-', ' '))

df_track_obj = reusable()
df_track = df_track_obj.dropColumns(df_track, ['_rescued_data'])
df_track = df_track_obj.dropDuplicates(df_track, ['track_id'])

In [0]:
query = (df_track.writeStream.format("delta")
    .outputMode("append")
    .option("checkpointLocation","abfss://silver@storageazureshan.dfs.core.windows.net/DimTrack/checkpoint_write")
    .trigger(availableNow=True)
    .start("abfss://silver@storageazureshan.dfs.core.windows.net/DimTrack/data"))
query.awaitTermination()

In [0]:
   display(spark.read.format("delta").load("abfss://silver@storageazureshan.dfs.core.windows.net/DimTrack/data"))

track_id,track_name,artist_id,album_name,duration_sec,release_date,updated_at,_rescued_data,durationFlag
1,Stand alone intangible encryption,434,Technology Album,234,2021-02-09,2025-09-12T19:49:55.000Z,null,medium
2,Programmable contextually based forecast,418,Current Album,110,2023-09-30,2025-09-26T19:49:55.000Z,null,low
3,Enhanced tertiary Internet solution,35,Born Album,195,2022-01-11,2025-09-11T19:49:55.000Z,null,medium
4,Mandatory user facing framework,54,Ball Album,167,2023-06-30,2025-10-02T19:49:55.000Z,null,medium
5,Multi layered needs based concept,340,Doctor Album,137,2022-06-08,2025-10-02T19:49:55.000Z,null,low
6,Synchronized high level complexity,384,City Album,250,2021-10-16,2025-09-22T19:49:55.000Z,null,medium
7,Total human resource structure,317,Parent Album,194,2022-08-02,2025-10-07T19:49:55.000Z,null,medium
8,Exclusive multimedia matrices,37,Owner Album,206,2022-06-22,2025-10-02T19:49:55.000Z,null,medium
9,Digitized multimedia hardware,165,Hand Album,342,2023-12-03,2025-10-02T19:49:55.000Z,null,high
10,Phased dynamic utilization,15,Hair Album,199,2023-12-17,2025-10-07T19:49:55.000Z,null,medium


In [0]:
df_raw = spark.read.format("parquet").load("abfss://bronze@storageazureshan.dfs.core.windows.net/DimTrack")
display(df_raw)

track_id,track_name,artist_id,album_name,duration_sec,release_date,updated_at
1,Stand-alone intangible encryption,434,Technology Album,234,2021-02-09,2025-09-12T19:49:55.000Z
2,Programmable contextually-based forecast,418,Current Album,110,2023-09-30,2025-09-26T19:49:55.000Z
3,Enhanced tertiary Internet solution,35,Born Album,195,2022-01-11,2025-09-11T19:49:55.000Z
4,Mandatory user-facing framework,54,Ball Album,167,2023-06-30,2025-10-02T19:49:55.000Z
5,Multi-layered needs-based concept,340,Doctor Album,137,2022-06-08,2025-10-02T19:49:55.000Z
6,Synchronized high-level complexity,384,City Album,250,2021-10-16,2025-09-22T19:49:55.000Z
7,Total human-resource structure,317,Parent Album,194,2022-08-02,2025-10-07T19:49:55.000Z
8,Exclusive multimedia matrices,37,Owner Album,206,2022-06-22,2025-10-02T19:49:55.000Z
9,Digitized multimedia hardware,165,Hand Album,342,2023-12-03,2025-10-02T19:49:55.000Z
10,Phased dynamic utilization,15,Hair Album,199,2023-12-17,2025-10-07T19:49:55.000Z


## DimDate


In [0]:
df_date = spark.readStream.format("cloudFiles")\
    .option("cloudFiles.format","parquet")\
    .option("cloudFiles.schemaLocation","abfss://silver@storageazureshan.dfs.core.windows.net/DimDate/checkpoint")\
    .option("cloudFiles.schemaEvolutionMode","addNewColumns")\
    .load("abfss://bronze@storageazureshan.dfs.core.windows.net/DimDate")

In [0]:
df_date_obj = reusable()
df_date = df_date_obj.dropColumns(df_date, ['_rescued_data'])
df_date = df_date_obj.dropDuplicates(df_date, ['date_key'])

In [0]:
query = (df_date.writeStream.format("delta")
    .outputMode("append")
    .option("checkpointLocation","abfss://silver@storageazureshan.dfs.core.windows.net/DimDate/checkpoint_write")
    .trigger(availableNow=True)
    .start("abfss://silver@storageazureshan.dfs.core.windows.net/DimDate/data"))
query.awaitTermination()

**FactStream**

In [0]:
df_fact = spark.readStream.format("cloudFiles")\
    .option("cloudFiles.format","parquet")\
    .option("cloudFiles.schemaLocation","abfss://silver@storageazureshan.dfs.core.windows.net/FactStream/checkpoint")\
    .option("cloudFiles.schemaEvolutionMode","addNewColumns")\
    .load("abfss://bronze@storageazureshan.dfs.core.windows.net/FactStream")

In [0]:
df_fact_obj = reusable()
df_fact = df_fact_obj.dropColumns(df_fact, ['_rescued_data'])

In [0]:
query = (df_fact.writeStream.format("delta")
    .outputMode("append")
    .option("checkpointLocation","abfss://silver@storageazureshan.dfs.core.windows.net/FactStream/checkpoint_write")
    .trigger(availableNow=True)
    .start("abfss://silver@storageazureshan.dfs.core.windows.net/FactStream/data"))
query.awaitTermination()

In [0]:
spark.sql("""CREATE TABLE IF NOT EXISTS spotify_cata.silver.dimtrack
USING DELTA
LOCATION 'abfss://silver@storageazureshan.dfs.core.windows.net/DimTrack/data'""")

spark.sql("""CREATE TABLE IF NOT EXISTS spotify_cata.silver.dimdate
USING DELTA
LOCATION 'abfss://silver@storageazureshan.dfs.core.windows.net/DimDate/data'""")

spark.sql("""CREATE TABLE IF NOT EXISTS spotify_cata.silver.factstream
USING DELTA
LOCATION 'abfss://silver@storageazureshan.dfs.core.windows.net/FactStream/data'""")

DataFrame[]